In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/churn_clean.csv')
print(df.shape)
print(df['Churn'].value_counts())

(7032, 29)
Churn
0    5163
1    1869
Name: count, dtype: int64


In [2]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"Train churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate: {y_test.mean()*100:.2f}%")

Train size: (5625, 28)
Test size: (1407, 28)
Train churn rate: 26.58%
Test churn rate: 26.58%


In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaling done")

Scaling done


In [4]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'AUC': roc_auc_score(y_test, y_prob),
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)
    }
    print(f"{name}: AUC={results[name]['AUC']:.4f} F1={results[name]['F1']:.4f}")

Logistic Regression: AUC=0.8374 F1=0.5793
Random Forest: AUC=0.8175 F1=0.5587
XGBoost: AUC=0.8114 F1=0.5754
LightGBM: AUC=0.8305 F1=0.5694


In [5]:
results_df = pd.DataFrame(results).T.round(4)
print(results_df.sort_values('AUC', ascending=False))

                        AUC      F1  Precision  Recall
Logistic Regression  0.8374  0.5793     0.6358  0.5321
LightGBM             0.8305  0.5694     0.6123  0.5321
Random Forest        0.8175  0.5587     0.6288  0.5027
XGBoost              0.8114  0.5754     0.6023  0.5508


In [6]:
best_model = XGBClassifier(random_state=42, eval_metric='logloss')
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Positives (caught churners): {cm[1][1]}")
print(f"False Negatives (missed churners): {cm[1][0]}")
print(f"False Positives (wrong alerts): {cm[0][1]}")

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.84      0.87      0.86      1033
       Churn       0.60      0.55      0.58       374

    accuracy                           0.78      1407
   macro avg       0.72      0.71      0.72      1407
weighted avg       0.78      0.78      0.78      1407


Confusion Matrix:
[[897 136]
 [168 206]]

True Positives (caught churners): 206
False Negatives (missed churners): 168
False Positives (wrong alerts): 136


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    X, y, cv=cv, scoring='roc_auc'
)
print(f"5-Fold CV AUC scores: {cv_scores.round(4)}")
print(f"Mean AUC: {cv_scores.mean():.4f}")
print(f"Std AUC: {cv_scores.std():.4f}")

5-Fold CV AUC scores: [0.8236 0.8302 0.8229 0.8203 0.8169]
Mean AUC: 0.8228
Std AUC: 0.0044


In [9]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print("Top 10 most important features:")
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': np.abs(shap_values).mean(axis=0)
}).sort_values('Importance', ascending=False)

print(feature_importance.head(10))
feature_importance.to_csv('../data/processed/feature_importance.csv', index=False)

Top 10 most important features:
                           Feature  Importance
4                           tenure    0.733100
19               Contract_Two year    0.661299
24                ChargePerService    0.454692
18               Contract_One year    0.341366
14                  MonthlyCharges    0.319534
15                    TotalCharges    0.301569
23                AvgMonthlyCharge    0.293148
16     InternetService_Fiber optic    0.202647
21  PaymentMethod_Electronic check    0.191925
6                    MultipleLines    0.167275
